##  Word Analogy Function

### Intuition
When Word2Vec is trained, each word is mapped to a point in high-dimensional space.
The geometry of that space captures **meaning** — specifically, relationships between
words appear as consistent *directions* (vectors).

For example:
- The direction **man → king** is roughly the same as **woman → queen**
- The direction **Paris → France** is roughly the same as **Berlin → Germany**

This means the *difference* between two word vectors encodes their relationship.

---

### The Math
Given three words **A, B, A\***, we want to find **B\*** such that:

> *"A is to B as A\* is to B\*"*

We compute:

$$\vec{B^*} \approx \vec{B} - \vec{A} + \vec{A^*}$$

Then we search the vocabulary for the word whose vector is **closest** to this target
(using cosine similarity), excluding the three input words from the results.

---

### Concrete Example

$$\vec{\text{king}} - \vec{\text{man}} + \vec{\text{woman}} \approx \vec{\text{queen}}$$

Step by step:
1. $\vec{\text{king}} - \vec{\text{man}}$ → extracts the **"royalty"** direction, stripping away the male component
2. $+ \vec{\text{woman}}$ → adds the **female** component back
3. The result lands near $\vec{\text{queen}}$ in the embedding space

This is never explicitly learned — the geometry **emerges naturally** from co-occurrence
patterns in the training text.

In [1]:
from load import load_embeddings_and_vocab

embeddings, word2idx, idx2word = load_embeddings_and_vocab()

In [6]:
import torch
import torch.nn.functional as F

def word_analogy(a, b, a_star, embeddings=embeddings, word2idx=word2idx, idx2word=idx2word, top_k=5):
    """
    Solves: word_a is to word_b as word_astar is to ?
    Formula: target = vec(B) - vec(A) + vec(A*)
    """

    a = a.lower()
    b = b.lower()
    a_star = a_star.lower()

    for word in [a, b, a_star]:
        if word not in word2idx:
            raise ValueError(f"{word} not in vocabulary")

    # Get vectors
    vec_a = embeddings[word2idx[a]]
    vec_b = embeddings[word2idx[b]]
    vec_a_star = embeddings[word2idx[a_star]]

    # Compute analogy vector
    target_vec = vec_b - vec_a + vec_a_star

    # Normalize embeddings for cosine similarity
    embeddings_norm = F.normalize(embeddings, dim=1)
    target_vec = F.normalize(target_vec.unsqueeze(0), dim=1)

    # Compute cosine similarity
    similarities = torch.mm(target_vec, embeddings_norm.t()).squeeze(0)


    # Get top results
    top_indices = torch.topk(similarities, top_k + 3).indices  # extra to skip input words

    results = []
    for idx in top_indices:
        word = idx2word[idx.item()]
        if word not in [a, b, a_star]:  # skip input words
            results.append((word, similarities[idx].item()))
        if len(results) == top_k:
            break

    return results

In [7]:
# Analogy: King : Man :: Woman : ?
result = word_analogy("King", "Man", "Woman")

print("Analogy: 'King' is to 'Man' as 'Woman' is to ?")
for i, (word, score) in enumerate(result, 1):
    print(f"{i}. {word} -> {score:.4f}")

# Analogy: paris : france :: berlin : ?
result = word_analogy("Paris", "France", "Berlin")
print("\nAnalogy: 'Paris' is to 'France' as 'Berlin' is to ?")
for i, (word, score) in enumerate(result, 1):
    print(f"{i}. {word} -> {score:.4f}")

Analogy: 'King' is to 'Man' as 'Woman' is to ?
1. heroin -> 0.4842
2. wearing -> 0.4403
3. nurse -> 0.4390
4. vietnamese -> 0.4365
5. wounded -> 0.4352

Analogy: 'Paris' is to 'France' as 'Berlin' is to ?
1. djate -> 0.4596
2. vitoux -> 0.4552
3. dechy -> 0.4544
4. regine -> 0.4500
5. testud -> 0.4389
